# Fine-tuning de Gemma-7b-it con QLoRA/SFT — versión notebook

Mismo pipeline QLoRA/SFT del laboratorio Lecture04b, con `google/gemma-7b-it`
en vez de Qwen3-8B. Gemma ya está soportado de fábrica por la imagen base de
Hugging Face (transformers 4.42), así que este pipeline evita la cadena de
incompatibilidades de versiones que tocó resolver para Qwen3.

**Adaptado para correr sobre el contenedor JupyterHub de la clase 04**
(`quay.io/jupyter/tensorflow-notebook:x86_64-cuda-latest`, ver `docker-compose.yml`
de este lab). Esta es la versión notebook, celda por celda, de `train_gemma.py`
— mismo pipeline, mismos valores por defecto; úsala para explorar/depurar
interactivamente, y `train_gemma.py` para correr el entrenamiento completo de
una sola vez (por ejemplo desde una terminal de Jupyter).

## Qué cambia frente a la versión pensada para Vertex AI

1. **Faltan dependencias.** Esta imagen es la línea "tensorflow-notebook" de
   Jupyter Docker Stacks: trae TensorFlow + CUDA, pero **no** trae PyTorch ni
   el stack de Hugging Face (a diferencia de la línea "pytorch-notebook", que
   sí lo incluiría). La celda de instalación (abajo) se encarga de esto — el
   usuario `jovyan` tiene permisos de escritura sobre `/opt/conda`, así que no
   hace falta `sudo` ni `--user`.
2. **La salida ya no va a `AIP_MODEL_DIR`** (esa variable es de Vertex AI
   Training y no existe aquí). Los adaptadores se guardan por defecto en
   `/home/jovyan/labs/...` — la carpeta que el `docker-compose.yml` de este lab
   monta desde `$HOME/si7016-262` del host, así que sobreviven a que borres o
   reinicies el contenedor.
3. **Login de Hugging Face explícito e interactivo** (celda de abajo, con
   `getpass` — nunca hardcodees el token en el notebook).

## Requisito importante: Gemma es un modelo "gated"

1. Entra a https://huggingface.co/google/gemma-7b-it y acepta la licencia con tu cuenta de HF.
2. Genera un token de acceso en https://huggingface.co/settings/tokens.
3. Pégalo cuando la celda de login te lo pida (sección 1).

Sin esto, la descarga del modelo falla con un error 401/403 "gated repo".


## 0. Instalar dependencias que la imagen base no trae

In [ ]:
%pip install torch transformers peft trl bitsandbytes accelerate datasets huggingface_hub


## 1. Login en Hugging Face (interactivo, sin hardcodear el token)

In [ ]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
login(token=hf_token)
del hf_token  # no lo dejamos flotando en una variable más de lo necesario


## 2. Configuración

Valores por defecto idénticos a `train_gemma.py`. Ajusta lo que necesites
directamente aquí (en el `.py` sería vía `argparse`).


In [ ]:
import os
import types

args = types.SimpleNamespace(
    model_name="google/gemma-7b-it",       # alternativa más liviana: "google/gemma-2b-it"
    dataset_name="knkarthick/samsum",       # alternativa si samsum falla: "knkarthick/dialogsum"
    train_split="train[:2000]",
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=20,  # en el notebook arrancamos con una corrida corta de prueba; súbelo (o -1) para la corrida completa
    output_dir=os.environ.get("OUTPUT_DIR", "/home/jovyan/labs/gemma-7b-it-samsum-lora"),
)

print(f"Modelo base: {args.model_name}")
print(f"Dataset: {args.dataset_name} ({args.train_split})")
print(f"Salida de los adaptadores: {args.output_dir}")


## 3. Verificar GPU

In [ ]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: no se detectó GPU. Revisa que el contenedor se haya levantado "
          "con --gpus all / runtime: nvidia (ver docker-compose.yml de este lab) "
          "y que `nvidia-smi` funcione dentro del contenedor.")


## 4. Cargar el modelo base en 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    args.model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(args.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Memoria GPU ocupada tras cargar el modelo (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "N/A")


## 5. Adaptadores LoRA

Mismos `target_modules` que en el notebook original — Gemma usa la misma
convención de nombres de proyecciones que Llama/Qwen.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=args.lora_r,
    lora_alpha=args.lora_alpha,
    lora_dropout=args.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Dataset + formateo con chat template

Nota: a diferencia de Qwen3, la plantilla de chat de Gemma no tiene modo
"thinking", así que NO pasamos `enable_thinking` aquí (evita cualquier duda
sobre kwargs específicos de un solo modelo).


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(args.dataset_name, split=args.train_split)

def format_example(example):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{example['dialogue']}"},
        {"role": "assistant", "content": example["summary"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print("Ejemplo formateado:\n", dataset[0]["text"][:400])


## 7. Entrenamiento con `SFTTrainer`

Los checkpoints intermedios se quedan en el disco efímero del contenedor
(`/tmp/...`) — solo los adaptadores finales (sección 8) se guardan en
`args.output_dir`, que sí persiste en el host vía el volumen montado.


In [ ]:
from trl import SFTConfig, SFTTrainer

local_ckpt_dir = "/tmp/gemma-7b-it-samsum-lora-ckpts"
sft_config = SFTConfig(
    output_dir=local_ckpt_dir,
    per_device_train_batch_size=args.per_device_train_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    num_train_epochs=args.num_train_epochs,
    max_steps=args.max_steps,
    learning_rate=args.learning_rate,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
)
trainer.train()


## 8. Guardar los adaptadores LoRA en el volumen montado

In [ ]:
os.makedirs(args.output_dir, exist_ok=True)
trainer.save_model(args.output_dir)
tokenizer.save_pretrained(args.output_dir)
print(f"Adaptadores guardados en: {args.output_dir}")
print("(esa carpeta vive dentro del volumen montado -sigue disponible en "
      "$HOME/si7016-262 del host aunque borres el contenedor)")


## Notas finales

- Para la corrida completa (no la prueba rápida de 20 pasos), vuelve a la
  sección 2 y pon `max_steps=-1` (así se respeta `num_train_epochs=3`
  completas).
- Este mismo pipeline existe como script plano en `train_gemma.py`, útil para
  lanzarlo de una sola vez desde una terminal (`python train_gemma.py --hf_token ...`)
  en vez de correrlo celda por celda.
- `requirements.txt` (en esta misma carpeta) lista las dependencias que la
  celda 0 instaló — útil si prefieres instalarlas por fuera del notebook
  (`pip install -r requirements.txt`) antes de abrirlo.
